# 第 7 关 · 控制者

固定位点 / 温度 / logits 偏置 / 跨分子。
这是五问第 3 问，也是逆向设计真正值钱的地方。

**BOSS：固定位点 100% 保住，且蛋白对照跑通**

---

**这一关有 7 个部分：**

0. 准备
1. 自由设计：什么都不限制
2. 固定位点：锁住环区，重设计茎区
3. 反过来：锁住茎区，重设计环区
4. logits 偏置：全局禁用一个碱基
5. 同一骨架，多条设计
6. 跨分子：同一套代码跑蛋白

## 部分 0 · 准备

用第 6 关训好的权重。

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "minif").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("仓库根：", ROOT)

from minif.plotting import setup
plt = setup()

In [ ]:
import os
import numpy as np
import torch

from minif import data, engine, features as F, pairing, paths, scoreboard
from minif.model import MiniMPNN
from minif.plotting import tidy, PALETTE

MOL = "rna"
CKPT = paths.run_file("level6.pt")
if not os.path.exists(CKPT):
    raise SystemExit("找不到 %s —— 先把第 6 关跑完" % CKPT)

va = data.load(MOL, "val")
model = MiniMPNN(d_node=F.node_dim(MOL), d_edge=F.edge_dim(MOL),
                 n_letters=F.n_letters(MOL), d_model=128, d_hidden=128, n_enc=3, n_dec=3)
model.load_state_dict(torch.load(CKPT)); model.eval()

r = max(va, key=lambda x: (x["partner"] >= 0).sum())
seq, partner, L = r["seq"], r["partner"], len(r["seq"])
dots = "".join("(" if 0 <= p and p > i else (")" if p >= 0 else ".") for i, p in enumerate(partner))
print("%s   %d nt，%d 对配对" % (r["id"], L, int((partner >= 0).sum()//2)))
print("配对  ", dots)
print("天然  ", seq)

## 部分 1 · 自由设计：什么都不限制

In [ ]:
free = engine.design(model, r, MOL, temperature=0.3)
print("自由  ", free, "  与天然 %.0f%%" % (100*sum(a==b for a,b in zip(seq, free))/L))
pairing.summarize(free, partner, "        ")

## 部分 2 · 固定位点：锁住环区，重设计茎区

**功能位点通常在环上**（配体结合口袋、催化中心）。锁住它们，让模型重排茎区。

能直接做到这件事，是因为训练时用的是随机解码顺序——
模型见过各种「已知一部分补其余」的局面。固定位点要排在解码顺序**最前面**，
否则它们的信息传不给后面的位置。

In [ ]:
loop = [i for i in range(L) if partner[i] < 0]
fixed = {int(p): int(F.seq_to_idx(seq, MOL)[p]) for p in loop}
out = engine.design(model, r, MOL, temperature=0.3, fixed=fixed)
held = sum(1 for p, a in fixed.items() if F.seq_to_idx(out, MOL)[p] == a) / max(len(fixed), 1)

print("锁定  ", "".join("^" if i in fixed else " " for i in range(L)), " (%d 个)" % len(fixed))
print("天然  ", seq)
print("结果  ", out)
print("\n固定位点保住 %.0f%%" % (100*held))
stem = [i for i in range(L) if i not in fixed]
print("非固定位置与天然相同 %.0f%%" % (100*np.mean([out[i]==seq[i] for i in stem])) if stem else "")
pairing.summarize(out, partner, "        ")
scoreboard.record(7, fixed_held=held)

## 部分 3 · 反过来：锁住茎区，重设计环区

茎区被锁死之后，环区怎么排？这一边应该自由得多——
也正说明环区的序列约束本来就弱。

In [ ]:
fixed2 = {i: int(F.seq_to_idx(seq, MOL)[i]) for i in range(L) if partner[i] >= 0}
out2 = engine.design(model, r, MOL, temperature=0.3, fixed=fixed2)
print("锁定  ", "".join("^" if i in fixed2 else " " for i in range(L)), " (%d 个)" % len(fixed2))
print("天然  ", seq)
print("结果  ", out2)

## 部分 4 · logits 偏置：全局禁用一个碱基

给某个碱基的分数加一个极大的负数，等于禁用它。
禁掉 G 之后 GC 对只能变成 AU/UA——看配对有效率掉多少。**这是加约束的代价。**

In [ ]:
bias = torch.zeros(F.n_letters(MOL)); bias[F.alphabet(MOL).index("G")] = -1e4
nog = engine.design(model, r, MOL, temperature=0.3, bias=bias)
print("禁用 G：", nog)
print("还有 G 吗：", "G" in nog)
pairing.summarize(nog, partner, "        ")
v_free, _ = pairing.pair_validity(free, partner)
v_nog, _ = pairing.pair_validity(nog, partner)
print("\n配对有效率 %.0f%% -> %.0f%%，这就是禁用一个碱基的代价"
      % (100*v_free, 100*v_nog))

## 部分 5 · 同一骨架，多条设计

逆向设计的产出不是一条序列，是一批候选。看它们彼此差多少、配对还对不对。

In [ ]:
outs = [engine.design(model, r, MOL, temperature=0.5) for _ in range(5)]
for o in outs:
    v, _ = pairing.pair_validity(o, partner)
    print("  %s  与天然 %2.0f%%  配对有效 %3.0f%%"
          % (o, 100*sum(a==b for a,b in zip(seq, o))/L, 100*v))
pw = [1 - sum(a==b for a,b in zip(outs[x], outs[y]))/L
      for x in range(5) for y in range(x+1, 5)]
print("\n5 条之间平均差异 %.0f%%" % (100*np.mean(pw)))

## 部分 6 · 跨分子：同一套代码跑蛋白

改的只有一个参数：`mol = "rna"` -> `"protein"`。
背后自动跟着变的只有三样：字母表 4→20、骨架原子、赝扭转角。
kNN、RBF、相对位置、消息传递、随机顺序自回归、固定位点——**一行没动**。

这就是 gRNAde 和 ProteinMPNN 是同一范式的具体含义。

In [ ]:
if not os.path.exists(data.path_for("protein")):
    print("先在终端跑：python prepare_data.py --mol protein")
    print("下完回来再跑这一格")
else:
    ptr = data.load("protein", "train"); pva = data.load("protein", "val")
    data.describe(ptr, "蛋白训练集："); data.describe(pva, "蛋白验证集：")
    pm = MiniMPNN(d_node=F.node_dim("protein"), d_edge=F.edge_dim("protein"),
                  n_letters=F.n_letters("protein"), d_model=128, d_hidden=128,
                  n_enc=3, n_dec=3)
    pr = engine.fit(pm, ptr, pva, "protein", epochs=40, lr=1e-3, noise=0.02,
                    autoregressive=True, log_every=10,
                    save=paths.run_file("level7_protein.pt"), eval_pairs=False)
    scoreboard.record(7, protein_recovery=pr)

In [ ]:
lv = scoreboard.read()["levels"]
print("RNA   字母表 %2d  边特征 %3d 维  恢复率 %s"
      % (F.n_letters("rna"), F.edge_dim("rna"),
         "%.1f%%" % (100*lv["6"]["recovery"]) if lv.get("6", {}).get("recovery") else "—"))
print("蛋白  字母表 %2d  边特征 %3d 维  恢复率 %s"
      % (F.n_letters("protein"), F.edge_dim("protein"),
         "%.1f%%" % (100*lv["7"]["protein_recovery"]) if lv.get("7", {}).get("protein_recovery") else "—"))
print()
print("两边的基准线不一样：RNA 瞎猜 25%，蛋白瞎猜 5%。")
print("所以蛋白的恢复率数字看着低，相对基准线的提升反而更大。")

In [ ]:
!cd .. && python game.py check 7

# 收口

技术部分写进 `notes/level7.md`（**这个仓库是公开的，只写技术**）：

- 五问的答案
- 「我这个版本和官方差在哪」——README 里列了五条，用自己跑出来的数字补充
- 「换一种分子要改哪几行」：字母表 4→20、骨架原子、赝扭转角、
  边特征原子对数 16→25，其余一行没动

研究方向相关的东西——核糖开关双构象切换那个空白、你打算怎么做——
写进你自己的**私有** vault，不要放进这个公开仓库。那是还没发表的想法。

最后：口述一遍，这个模型我怎么实现的，五分钟，掐表。